In [2]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install webdriver_manager

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [5]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import pandas as pd
import time

In [6]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [7]:
pip install re

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement re (from versions: none)
ERROR: No matching distribution found for re


In [8]:
import re

In [81]:
def get_atr_vacancy (driver):

    try:
        wage_elem = driver.find_element(By.CLASS_NAME, 'resume-block__salary')
        wage_text = wage_elem.text
        wage = re.sub(r'\D', '', wage_text)
    except:
        wage = ""

    try:
        age_elem = driver.find_element(By.CSS_SELECTOR, "[data-qa='resume-personal-age']")
        age_text = age_elem.text
        age = re.sub(r'\D', '', age_text)
    except:
        age = ""

    try:
        gender_elem = driver.find_element(By.CSS_SELECTOR, "[data-qa='resume-personal-gender']")
        gender = gender_elem.text
    except:
        gender = ""

    try:
        metro_elem = driver.find_element(By.CSS_SELECTOR, "[data-qa='resume-personal-metro']")
        match = re.search(r"м\.\s*(.+)",metro_elem.text)
        metro = match.group(1)
    except:
        metro = ""

    try:
        experience_elem = driver.find_element(By.CSS_SELECTOR, "[data-qa='resume-block-experience']")
        experience_elem = experience_elem.find_element(By.CSS_SELECTOR, "[class^='resume-block__title-text resume-block__title-text_sub']")
        
        pattern = r"(\d+)\s*(лет|год[а]?|месяц[а]?)"

        matches = re.findall(pattern, experience_elem.text)
        experience = 0

        for match in matches:
            i = int(match[0])
            period = match[1]

            if period in ["год", "года", "лет"]:
                experience += 12*i
            elif period in ["месяц", "месяца"]:
                experience += i

    except:
        experience = ""

    return wage, age, gender, metro, experience

In [85]:
service = Service()
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options, service = service)

wages = []
ages = []
genders = []
metroes = []
experiences = []

driver.get("https://hh.ru/resumes/administrator_na_resepshn")

if "Not Found" in driver.title:
  print("Can't found link")
else:
  for _ in range(10):

    vacancies = driver.find_elements(By.CSS_SELECTOR, "[class^='column-content']")
    for vacancy in vacancies:
      if "Не ищет работу" in vacancy.text:
        print("скипнули вакансию",vacancy.text[:20])
        continue
      else:
        vacancy = vacancy.find_element(By.CSS_SELECTOR, "[class^='magritte-link']")
        try:
          driver.execute_script("arguments[0].scrollIntoView(true);", vacancy)
          time.sleep(2)

          WebDriverWait(driver, 10).until(EC.element_to_be_clickable(vacancy))
          vacancy.click()
          time.sleep(4)

          try:
            driver.switch_to.window(driver.window_handles[1])
            time.sleep(2)

            wage, age, gender, metro, experience = get_atr_vacancy(driver)
            
            wages.append(wage)
            ages.append(age)
            genders.append(gender)
            metroes.append(metro)
            experiences.append(experience)

            driver.close()

            driver.switch_to.window(driver.window_handles[0])
            time.sleep(2)
          except:
            driver.switch_to.window(driver.window_handles[1])
            time.sleep(2)
            
            driver.close()

            driver.switch_to.window(driver.window_handles[0])
            time.sleep(2)

        except Exception as e:
          print(f"Ошибка при клике на элемент: {e}")


    button = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "[data-qa*='pager-next']"))
    )
    driver.execute_script("arguments[0].scrollIntoView(true);", button)
    button.click()
    time.sleep(4)

скипнули вакансию Администратор
38 лет
скипнули вакансию Администратор на рес
скипнули вакансию Секретарь на ресепшн
скипнули вакансию Администратор, секре
скипнули вакансию Администратор на рес
скипнули вакансию администратор на рес
скипнули вакансию Администратор
24 год
скипнули вакансию Продавец-консультант
скипнули вакансию Секретарь, администр
скипнули вакансию Секретарь на ресепшн
скипнули вакансию Секретарь на ресепшн
скипнули вакансию Администратор офиса

скипнули вакансию Секретарь
29 лет  • 
скипнули вакансию Секретарь на ресепшн
скипнули вакансию Администратор на рес
скипнули вакансию администратор, менед
скипнули вакансию Менеджер, администра
скипнули вакансию Администратор
25 лет
скипнули вакансию Администратор на рес
скипнули вакансию Секретарь
32 года  •
скипнули вакансию Секретарь на ресепшн
скипнули вакансию Администратор на рес
скипнули вакансию Секретарь-администра
скипнули вакансию Администратор на рес
скипнули вакансию Секретарь на ресепшн
скипнули вакансию Админис

In [86]:
df = pd.DataFrame({
    'wage': wages,
    'age': ages,
    'gender': genders,
    'metro': metroes,
    'experience': experiences,
})

df.to_csv('C:/Users/User/Desktop/проект НоД/vacancies.csv', index=False, encoding='cp1251')

In [87]:
df

,wage,age,gender,metro,experience
0,40000,33,Женщина,,157
1,30000,29,Женщина,Братиславская,127
2,30000,30,Мужчина,Парк Победы,150
3,,57,Женщина,Отрадное,276
4,40000,29,Женщина,Новогиреево,113
...,...,...,...,...,...
157,30000,47,Женщина,,114
158,40000,58,Женщина,Университет,188
159,25000,35,Мужчина,,211
160,27000,45,Женщина,Выхино,275
